### 1. Removing Structural Noise
The raw dataset contains a column named 'Unnamed : 0',which apperrs to be a residual index from a previous file save

**Critical issue:** This column assigns a unique number to every single row. So, as long as this row exists pandas will see every row as a unique row making it imopossible to detect duplicate. We must remove it to perform accurate data integrity checks.

In [2]:
import pandas as pd 

In [14]:
df = pd.read_csv('../data/ds_salaries.csv')

In [4]:
#check the noise column exits
if 'Unnamed: 0' in df.columns:
    print("found the column named 'Unnamed: 0',Removing it... ")
    df.drop('Unnamed: 0',axis = 1,inplace = True)
    print("Column removed succesfully")
else:
    print("No column found with name 'Unnamed: 0' , data might be already cleaned ")

found the column named 'Unnamed: 0',Removing it... 
Column removed succesfully


In [5]:
#verifying the dataset using shape and head
print('current dataset shape :) {df.shape}')
print('\n\n')
display(df.head(3))

current dataset shape :) {df.shape}





,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M


### 2. The hidden duplicate hunt 
Now the artificial index column is dropped . we can now identify true duplicates

**why this matters :**
Duplicate rows in survey data usually represent data entry errors or resubmissions. Keeping them creates **bias**—it makes certain salaries or job titles appear more common than they really are, which artificially lowers the p-values in our statistical significance tests later.


**Action:** Identifying  and removing these duplicates to ensure each row represents a unique data point.

In [6]:
#finding duplicate count
duplicate_count = df.duplicated().sum()
print(f"duplicates found : {duplicate_count}")

duplicates found : 42


In [7]:
#sample of duplicate date
if duplicate_count > 0:
    print('\nsample of duplicate rows :')
    display(df[df.duplicated].head())


sample of duplicate rows :


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
217,2021,MI,FT,Data Scientist,76760,EUR,90734,DE,50,DE,L
256,2021,MI,FT,Data Engineer,200000,USD,200000,US,100,US,L
331,2022,SE,FT,Data Analyst,90320,USD,90320,US,100,US,M
332,2022,SE,FT,Data Analyst,112900,USD,112900,US,100,US,M
333,2022,SE,FT,Data Analyst,90320,USD,90320,US,100,US,M


In [8]:
#removing duplicates
df.drop_duplicates(inplace = True)
print(f'\n duplicates are removed , the final shape of the dataset is {df.shape}')


 duplicates are removed , the final shape of the dataset is (565, 11)


### 3. Feature engineering: Standardizing job roles
The dataset contains 50 unique job titles , which creates high cradinality and makes visualization difficult

**Action:** We will implement a mapping function to categorize these titles into 4 standardized industry roles: **Data Scientist, Data Engineer, Data Analyst, and Machine Learning Engineer**. 
This simplification allows for more robust statistical comparisons (e.g., "Do ML Engineers earn significantly more than Data Analysts?").

In [9]:
# function to map the 50 job titles to 4 categories
def categorize_job_title(title):
    title = title.lower()

    if 'machine learning' in title or 'ml' in title or 'computer vision' in title or 'nlp' in title or 'ai' in title:
        return 'Machine Learning Engineer'
    elif 'data scientist' in title or 'science' in title:
        return 'Data Scientist'
    elif 'data engineer' in title or 'architect' in title or 'etl' in title:
        return 'Data Engineer'
    elif 'analyst' in title or 'analytics' in title:
        return 'Data Analyst'
    else:
        return 'Other'

# apply the createdd function
df['job_category'] = df['job_title'].apply(categorize_job_title)

# to verify
print('\nNew job category distribution:')
print(df['job_category'].value_counts())


New job category distribution:
job_category
Data Scientist               179
Data Engineer                160
Data Analyst                 120
Machine Learning Engineer     84
Other                         22
Name: count, dtype: int64


In [ ]:
#empty cell

### 4.Feature Engineering : Salary Tiers
To faciliate cohort analysis we discretize the continuous salary into 5 ordinal tiers

**Action:** Creating a `salary_band` column using `pd.cut` with defined edges at 60k, 100k, 150k, and 250k.

In [10]:
import numpy as np
df['salary_band'] = pd.cut(
    df['salary_in_usd'],
    bins = [0,60000,100000,150000,250000,np.inf],
    labels = ['Entry(<60k)','junior(60k - 100k)','MId(100k - 150k)','Senior(150k - 250k)','Elite(250k>)']
)

print("salary band distribution:")
print(df['salary_band'].value_counts().sort_index())

salary band distribution:
salary_band
Entry(<60k)            141
junior(60k - 100k)     149
MId(100k - 150k)       142
Senior(150k - 250k)    117
Elite(250k>)            16
Name: count, dtype: int64


In [11]:
#empty ccell

### 5. Saving the Trusted Data Layer
We have successfully cleaned the data, removed duplicates, and engineered features for analysis. We will now save this as a new file, `ds_salaries_cleaned.csv`.

**Next Step:** Notebook 03 will load this specific file to perform Univariate Analysis.

In [12]:
df.to_csv('ds_salaries_cleaned.csv', index=False)

print("Data saved to 'ds_salaries_cleaned.csv'.")
print(f"Final Dataset Shape: {df.shape}")

Data saved to 'ds_salaries_cleaned.csv'.
Final Dataset Shape: (565, 13)


In [ ]:
#notebook -2 end